In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import tensorflow as tf
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [4]:
def load_severity_paths(dataset_dir):

    paths=[]
    labels=[]

    for disaster in DISASTER_TYPES:
        for severity in SEVERITY_LEVELS:

            folder=os.path.join(dataset_dir,disaster,severity)

            if not os.path.exists(folder):
                continue

            for file in os.listdir(folder):

                if file.lower().endswith((".jpg",".jpeg",".png")):

                    paths.append(os.path.join(folder,file))
                    labels.append(SEVERITY_LABEL[severity])

    return np.array(paths),np.array(labels)

In [7]:
DISASTER_TYPES = ["earthquake","flood","wildfire"]
SEVERITY_LEVELS = ["high","low","medium"]

SEVERITY_LABEL = {
    "high":0,
    "medium":1,
    "low":2
}

In [10]:
TRAIN_DIR = "/content/drive/MyDrive/disaster_final/train"
VAL_DIR = "/content/drive/MyDrive/disaster_final/validation"
TEST_DIR = "/content/drive/MyDrive/disaster_final/test"
train_paths,train_labels = load_severity_paths(TRAIN_DIR)
val_paths,val_labels = load_severity_paths(VAL_DIR)
test_paths,test_labels = load_severity_paths(TEST_DIR)

In [12]:
IMG_SIZE = 224
BATCH_SIZE = 16

def load_image(path, label):

    img = tf.io.read_file(path)

    img = tf.io.decode_image(img, channels=3)

    img.set_shape([None, None, 3])

    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    img = tf.cast(img, tf.float32) / 255.0

    return img, label

In [13]:
train_ds=tf.data.Dataset.from_tensor_slices((train_paths,train_labels))

train_ds=train_ds.map(load_image).shuffle(2000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds=tf.data.Dataset.from_tensor_slices((val_paths,val_labels))

val_ds=val_ds.map(load_image).batch(BATCH_SIZE)

In [15]:
stage2_model = load_model("/content/drive/MyDrive/classifier_model_V_Hera.keras")

In [16]:
for layer in stage2_model.layers:
    layer.trainable=False

In [18]:
features = stage2_model.layers[-2].output

x = Dense(128, activation="relu", name="severity_dense")(features)

severity_output = Dense(3, activation="softmax", name="severity_output")(x)

severity_model = Model(stage2_model.input, severity_output)

severity_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ severity_dense (Dense)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ severity_output (Dense)         │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,410,790 (16.83 MB)

 Trainable params: 33,283 (130.01 KB)

 Non-trainable params: 4,377,507 (16.70 MB)

In [19]:
severity_model.compile(
    optimizer=Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [20]:
severity_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

Epoch 1/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 685s 2s/step - accuracy: 0.3696 - loss: 1.1841 - val_accuracy: 0.6659 - val_loss: 0.8517
Epoch 2/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 297s 2s/step - accuracy: 0.6353 - loss: 0.8383 - val_accuracy: 0.7601 - val_loss: 0.6845
Epoch 3/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 314s 2s/step - accuracy: 0.7307 - loss: 0.6864 - val_accuracy: 0.7814 - val_loss: 0.6016
Epoch 4/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 396s 2s/step - accuracy: 0.7956 - loss: 0.5709 - val_accuracy: 0.7960 - val_loss: 0.5540
Epoch 5/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 297s 2s/step - accuracy: 0.7872 - loss: 0.5596 - val_accuracy: 0.8016 - val_loss: 0.5286
Epoch 6/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 303s 2s/step - accuracy: 0.8308 - loss: 0.4617 - val_accuracy: 0.8072 - val_loss: 0.5132
Epoch 7/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 306s 2s/step - accuracy: 0.8342 - loss: 0.4552 - val_accuracy: 0.8094 - val_loss: 0.5047
Epoch 8/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 321s 2s/step - accuracy: 0.8361 - loss: 0.4454 - val_accu

In [21]:
test_ds=tf.data.Dataset.from_tensor_slices((test_paths,test_labels))

test_ds=test_ds.map(load_image).batch(BATCH_SIZE)

severity_model.evaluate(test_ds)

29/29 ━━━━━━━━━━━━━━━━━━━━ 83s 3s/step - accuracy: 0.7129 - loss: 0.9305


[1.5167568922042847, 0.5151515007019043]

In [22]:
severity_model.save("/content/drive/MyDrive/Severity_classifier_model_V_Ares.keras")

print("Stage-3 model version Aressaved to Google Drive!")

Stage-3 model version Aressaved to Google Drive!
